In [3]:
# 1. Install necessary libraries
# We need 'pypdf' to read files and 'transformers' for the AI brain
!pip install -q gradio pypdf transformers torch

import gradio as gr
from pypdf import PdfReader
from transformers import pipeline

print("Loading AI Model... (this takes about 30 seconds)")
# We use a small, free model specifically made for Q&A
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

# --- Logic Functions ---

def extract_text_from_pdf(pdf_file):
    """Helper function to get text out of the uploaded PDF."""
    if pdf_file is None:
        return ""
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

def chatbot_response(pdf_file, question, style, history):
    """
    Main function to handle the chat.
    Args:
        pdf_file: The uploaded PDF
        question: User's question
        style: Selected style from dropdown (Extra Feature)
        history: Previous chat history
    """
    if pdf_file is None:
        return history + [[question, "⚠️ Please upload a PDF document first!"]]

    if question.strip() == "":
        return history

    # 1. Read the PDF
    context = extract_text_from_pdf(pdf_file)

    # 2. Limit text size (Small models can't read whole books at once)
    # We take the first 3000 characters for this demo.
    context_snippet = context[:3000]

    # 3. Get Answer from AI
    try:
        result = qa_pipeline(question=question, context=context_snippet)
        answer = result['answer']

        # 4. Apply "Style" (The Extra Feature)
        if style == "Formal":
            answer = f"According to the document: {answer}."
        elif style == "Enthusiastic":
            answer = f"Great question! I found this: {answer}!"

    except Exception as e:
        answer = "I couldn't find the answer in the first few pages of this document."

    # 5. Update Chat History
    history.append((question, answer))
    return history

# --- The UI (Customized) ---

# We use a specific theme to match "Make it your own" requirements
custom_theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="blue",
)

with gr.Blocks(theme=custom_theme, title="DocuBot Pro") as demo:

    # Header
    gr.Markdown(
        """
        # 📄 PDF Assistant Bot
        ### Upload a document and ask me anything about it!
        """
    )

    with gr.Row():
        # Left Column: Upload & Settings
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📂 Upload PDF", file_types=[".pdf"])

            # EXTRA FEATURE: Dropdown menu for mode selection
            style_select = gr.Dropdown(
                choices=["Standard", "Formal", "Enthusiastic"],
                value="Standard",
                label="🎭 Answer Style (Extra Feature)"
            )

            gr.Markdown("*Tip: This bot reads the first few pages of your file.*")

        # Right Column: Chat Interface
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", height=400)
            msg = gr.Textbox(label="💬 Ask a question...", placeholder="What is this document about?")
            clear = gr.ClearButton([msg, chatbot])

    # Interaction Handling
    # When user presses Enter in the textbox
    msg.submit(
        fn=chatbot_response,
        inputs=[pdf_input, msg, style_select, chatbot],
        outputs=[chatbot]
    )
    # Clear the textbox after sending
    msg.submit(lambda: "", None, msg)

# Launch
demo.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 3.3 MB/s eta 0:00:00


Loading AI Model... (this takes about 30 seconds)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Device set to use cpu
/tmp/ipython-input-2692327926.py:73: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, title="DocuBot Pro") as demo:
/tmp/ipython-input-2692327926.py:99: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat History", height=400)
/tmp/ipython-input-2692327926.py:99: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65180189b42ceda9e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
